# Clasificar textos con un LLM local: Ollama y Llama 3.2
## Data Mining - Doctorado UDP 2026
**Dr. Bastián González-Bustamante, Profesor Asociado** \
Noviembre 2026

**Sesión 13 · jueves 19 de noviembre de 2026 · Parte 2: demostración (2 de 2).**

**Este notebook corre solo en su computador**, no en Google Colab: el modelo de lenguaje se ejecuta con **Ollama**, un programa que se instala en su equipo. Por eso no tiene el botón "Abrir en Colab".

Usamos un LLM pequeño, **Llama 3.2 de 3.000 millones de parámetros** (`llama3.2`, una descarga de 2,0 GB), como **anotador**: le pedimos que clasifique los mismos mensajes de las Sesiones 12 y 13 como tóxicos o no tóxicos, y **validamos** sus respuestas contra la codificación humana.

| N.º | Sección | Presentación |
|---|---|---|
| 0 | Preparación: Ollama, el modelo y los datos | No aplica |
| 1 | *Zero-shot* con salida estructurada | *Prompting*; LLMs como anotadores |
| 2 | *Few-shot* con ejemplos del entrenamiento | *Prompting* |
| 3 | Cadena de razonamiento (CoT) | *Prompting* |
| 4 | Estabilidad del *prompt* | Reproducibilidad y estabilidad |
| 5 | Validación contra la codificación humana | Un protocolo de validación |
| 6 | Ejercicios | No aplica |

**Antes de empezar (una sola vez):**
1. Instalar Ollama desde https://ollama.com/download y abrir la aplicación (o ejecutar `ollama serve` en una terminal).
2. Descargar el modelo en una terminal: `ollama pull llama3.2`.
3. Instalar los paquetes del curso: `pip install -r requirements.txt` (incluye el cliente `ollama` para Python).

**Tiempo:** cada mensaje tarda una fracción de segundo con GPU y alrededor de uno a tres segundos en la CPU de un portátil. Con los tamaños de muestra por defecto, el notebook completo tarda unos 5 minutos con GPU y de 15 a 30 minutos solo con CPU. Puede reducir `N_MENSAJES`.

`random_state=86` y `seed=86` en todos los pasos aleatorios. **Advertencia de contenido:** algunos mensajes contienen lenguaje ofensivo.

### Entorno

La primera celda es la misma de los demás notebooks del curso: verifica las versiones. La segunda comprueba que **no** estamos en Colab, que Ollama está funcionando y que el modelo está descargado.

In [1]:
## Entorno: detectar Google Colab, instalar solo lo que falte y verificar las versiones
import re
import sys
from importlib import invalidate_caches
from importlib.metadata import version
from importlib.util import find_spec

## Paquetes que usa este notebook: {módulo: (nombre en pip, versión mínima soportada)}
PAQUETES = {
    "sklearn": ("scikit-learn", "1.6"),
    "pandas": ("pandas", "2.2"),
    "numpy": ("numpy", "2.0"),
    "matplotlib": ("matplotlib", "3.9"),
    "ollama": ("ollama", "0.4"),
}

EN_COLAB = "google.colab" in sys.modules
faltan = [pip for modulo, (pip, _) in PAQUETES.items() if find_spec(modulo) is None]
if faltan and EN_COLAB:
    ## En Colab se instalan solo los paquetes que faltan; los preinstalados no se actualizan
    get_ipython().run_line_magic("pip", "install -q " + " ".join(faltan))
    invalidate_caches()
    faltan = []
## En su computador no se instala nada: el entorno virtual se crea con requirements.txt
assert not faltan, f"Faltan paquetes: {', '.join(faltan)}. Instálelos con pip install -r requirements.txt."


def version_al_menos(paquete, minima):
    ## Compara los dos primeros números de la versión instalada con la mínima (p. ej., "1.6.1" >= "1.6")
    actual = tuple(int(n) for n in re.findall(r"\d+", version(paquete))[:2])
    return actual >= tuple(int(n) for n in minima.split("."))


print(f"Python {sys.version.split()[0]} | "
      + " | ".join(f"{pip} {version(pip)}" for pip, _ in PAQUETES.values())
      + (" | Google Colab" if EN_COLAB else " | computador local"))
assert sys.version_info >= (3, 11), "Este notebook requiere Python 3.11 o superior."
antiguos = [f"{pip} {version(pip)} (mínimo {minima})" for pip, minima in PAQUETES.values()
            if not version_al_menos(pip, minima)]
assert not antiguos, f"Versiones no soportadas: {'; '.join(antiguos)}. Actualice esos paquetes en su entorno."

Python 3.12.7 | scikit-learn 1.9.1 | pandas 3.0.6 | numpy 2.5.4 | matplotlib 3.11.2 | ollama 0.6.3 | computador local


In [2]:
## Solo local: Ollama debe estar funcionando y el modelo, descargado
import json
import urllib.request

import ollama

MODELO = "llama3.2"
if EN_COLAB:
    raise RuntimeError("Este notebook corre solo en su computador: Ollama no está disponible en Google Colab.")

try:
    with urllib.request.urlopen("http://localhost:11434/api/version", timeout=10) as respuesta:
        version_servidor = json.load(respuesta)["version"]
except OSError as error:
    raise RuntimeError("No se encontró Ollama. Abra la aplicación Ollama o ejecute `ollama serve` "
                       "en una terminal, y vuelva a ejecutar esta celda.") from error

disponibles = [m.model for m in ollama.list().models]
if not any(m.split(":")[0] == MODELO for m in disponibles):
    raise RuntimeError(f"Falta el modelo. Descárguelo en una terminal con: ollama pull {MODELO}")
detalle = ollama.show(MODELO)
print(f"Servidor Ollama {version_servidor} | cliente ollama {version('ollama')} | modelo {MODELO}: "
      f"{detalle.details.parameter_size} parámetros, cuantización {detalle.details.quantization_level}")

Servidor Ollama 0.40.7 | cliente ollama 0.6.3 | modelo llama3.2: 3.2B parámetros, cuantización Q4_K_M


In [3]:
## Importar librerías y fijar el estilo de los gráficos
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, cohen_kappa_score, precision_score, recall_score

plt.rcParams.update({"figure.dpi": 100, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.3, "axes.titlesize": 11, "axes.labelsize": 10})
AZUL, NARANJA, VERDE, GRIS = "#2a78d6", "#eb6834", "#1baf7a", "#8a8984"

### Datos

**▶ En vivo.** La misma partición de S12 y del notebook 5. Para que el notebook no tarde demasiado, clasificamos una **muestra estratificada de la prueba**: `N_MENSAJES = 100` para *zero-shot* y *few-shot*, y 50 para la cadena de razonamiento y la estabilidad.

In [4]:
## Patrón oro de protestas, partición de S12 y muestra estratificada de la prueba
URL_HF = "https://huggingface.co/datasets/bgonzalezbustamante/{}/resolve/main/{}"
protestas = pd.read_csv(URL_HF.format("toxicity-protests-ES", "goldstd_protests.csv"))
estratos = protestas["coder_1"].astype(str) + "_" + protestas["country"]
idx_train, idx_test = train_test_split(np.arange(len(protestas)), test_size=0.20,
                                       stratify=estratos, random_state=86)

N_MENSAJES = 100
N_PEQUEÑA = 50
muestra = (protestas.iloc[idx_test].groupby("coder_1", group_keys=False)
           .sample(frac=N_MENSAJES / len(idx_test), random_state=86).sort_index())
pequeña = (muestra.groupby("coder_1", group_keys=False)
           .sample(frac=N_PEQUEÑA / len(muestra), random_state=86).sort_index())
print(f"Muestra: {len(muestra)} mensajes ({muestra['coder_1'].mean():.0%} tóxicos) | "
      f"submuestra: {len(pequeña)} ({pequeña['coder_1'].mean():.0%} tóxicos)")

Muestra: 100 mensajes (54% tóxicos) | submuestra: 50 (54% tóxicos)


## 1. *Zero-shot* con salida estructurada

**▶ En vivo.** Usamos las mismas instrucciones del patrón oro y de González-Bustamante (2024): las definiciones de toxicidad de Perspective, en inglés. *Zero-shot* significa que el modelo no ve ningún ejemplo etiquetado.

**Salida estructurada.** En 2025 le pedíamos al modelo que respondiera "solo la categoría" y leíamos el texto que devolvía. Ahora le entregamos un **esquema JSON** (`format=`), y Ollama obliga al modelo a responder con un objeto `{"categoria": "TOXIC"}` o `{"categoria": "NONTOXIC"}`. Igual verificamos cada respuesta: lo que no se pueda leer queda como `REVISAR`, no se cuenta como "no tóxico".

**Reproducibilidad.** `temperature=0` y `seed=86`. Aun así, Barrie et al. (2024) advierten que el cómputo en GPU puede producir pequeñas diferencias entre ejecuciones.

In [5]:
## Instrucciones (las del patrón oro), esquema de la respuesta y una función para clasificar
SISTEMA = ("Classify the category of the comment as either TOXIC or NONTOXIC. "
           "TOXIC: Rude, disrespectful, or unreasonable comments that are likely to make someone leave "
           "the discussion or stop sharing their perspective. "
           "NONTOXIC: Civil or nice comments that are unlikely to discourage conversation.")
ESQUEMA = {"type": "object",
           "properties": {"categoria": {"type": "string", "enum": ["TOXIC", "NONTOXIC"]}},
           "required": ["categoria"]}
ESQUEMA_COT = {"type": "object",
               "properties": {"razonamiento": {"type": "string"},
                              "categoria": {"type": "string", "enum": ["TOXIC", "NONTOXIC"]}},
               "required": ["razonamiento", "categoria"]}
ETIQUETAS = {"TOXIC": 1, "NONTOXIC": 0}  # diccionario explícito, no el orden de frecuencia (error de 2025)
OPCIONES = {"temperature": 0, "seed": 86}


def clasificar(textos, sistema=SISTEMA, ejemplos=(), esquema=ESQUEMA, instruccion="Respond in JSON.",
               opciones=OPCIONES):
    respuestas, tiempo = [], time.perf_counter()
    for texto in textos:
        mensajes = [{"role": "system", "content": sistema}]
        for ej_texto, ej_categoria in ejemplos:
            mensajes += [{"role": "user", "content": f"text: {ej_texto}\n{instruccion}"},
                         {"role": "assistant", "content": json.dumps({"categoria": ej_categoria})}]
        mensajes.append({"role": "user", "content": f"text: {texto}\n{instruccion}"})
        salida = ollama.chat(model=MODELO, messages=mensajes, format=esquema, options=opciones)
        try:
            respuesta = json.loads(salida.message.content)
            categoria = respuesta["categoria"]
            assert categoria in ETIQUETAS
        except (json.JSONDecodeError, KeyError, TypeError, AssertionError):
            respuesta, categoria = {}, "REVISAR"
        respuestas.append({"categoria": categoria, **respuesta})
    tiempo = time.perf_counter() - tiempo
    print(f"{len(textos)} mensajes en {tiempo:.0f} s ({tiempo / len(textos):.2f} s por mensaje)")
    return pd.DataFrame(respuestas, index=list(range(len(textos))))


zero = clasificar(muestra["text"])
print(zero["categoria"].value_counts().to_dict())

100 mensajes en 17 s (0.17 s por mensaje)
{'TOXIC': 64, 'NONTOXIC': 36}


In [6]:
## El error de 2025: asignar las etiquetas según su frecuencia
## Si la categoría más frecuente fuera NONTOXIC, este mapeo invertiría las etiquetas sin avisar
respuestas_2025 = pd.Series(["NONTOXIC", "NONTOXIC", "TOXIC", "NONTOXIC", "TOXIC"])
orden = respuestas_2025.value_counts().index.tolist()
mapeo_2025 = {orden[0]: 1, orden[1]: 0}
print("Mapeo por frecuencia (2025):", mapeo_2025, "← NONTOXIC quedó como 1 (tóxico)")
print("Mapeo explícito (2026):     ", ETIQUETAS)

Mapeo por frecuencia (2025): {'NONTOXIC': 1, 'TOXIC': 0} ← NONTOXIC quedó como 1 (tóxico)
Mapeo explícito (2026):      {'TOXIC': 1, 'NONTOXIC': 0}


**Para recordar.**
- Con el esquema JSON, todas las respuestas son legibles: no hay que adivinar la categoría desde un texto libre.
- Las etiquetas se asignan con un **diccionario explícito**. El notebook de 2025 las asignaba según su frecuencia, lo que solo funcionaba porque "TOXIC" era la respuesta más común.

## 2. *Few-shot*: ejemplos tomados del entrenamiento

**▶ En vivo.** Agregamos a la conversación **cuatro ejemplos etiquetados** (dos tóxicos y dos no tóxicos), tomados al azar del **entrenamiento**, nunca de la prueba. En 2025 usábamos dos ejemplos inventados y extremos ("Eres un idiota" frente a "Muchas gracias por tu interesante comentario"), poco parecidos a los mensajes reales.

In [7]:
## Cuatro ejemplos del entrenamiento (2 tóxicos, 2 no tóxicos), cortos, elegidos con random_state=86
entrenamiento = protestas.iloc[idx_train]
cortos = entrenamiento[entrenamiento["text"].str.len() < 140]
ejemplos = pd.concat([cortos[cortos["coder_1"] == c].sample(2, random_state=86) for c in [1, 0]])
EJEMPLOS = [(t, "TOXIC" if c == 1 else "NONTOXIC") for t, c in zip(ejemplos["text"], ejemplos["coder_1"])]
for t, c in EJEMPLOS:
    sin_menciones = re.sub(r"@\w+", "@usuario", t)
    print(f"{c:8} | {sin_menciones[:110]}")

few = clasificar(muestra["text"], ejemplos=EJEMPLOS)
print(few["categoria"].value_counts().to_dict())

TOXIC    | @usuario @usuario #YoNoMarcho y vos tampoco por lo que sé, serás borracha pero no estúpida, más bien sos diabó
TOXIC    | #17AJuntosContraLaImpunidad yo voy a la marcha, al marcha de tu culo y mi garcha. Besitos
NONTOXIC | @usuario El agua se esta convirtiendo en lava, amigo. #QueSeVayanTodos
NONTOXIC | X que el 98,5 % de todos los políticos son lo peor que le pasó al país #QueSeVayanTodos


100 mensajes en 11 s (0.11 s por mensaje)
{'NONTOXIC': 56, 'TOXIC': 44}


## 3. Cadena de razonamiento (CoT)

**▶ En vivo.** Le pedimos al modelo que **primero razone** y luego dé la categoría, en un esquema con dos campos (`razonamiento` y `categoria`). Wei et al. (2022) mostraron que el razonamiento paso a paso ayuda sobre todo en modelos muy grandes (del orden de 100.000 millones de parámetros); nuestro modelo tiene 3.000 millones. Usamos la submuestra de 50 mensajes, porque cada respuesta es mucho más larga.

In [8]:
## Razonamiento paso a paso y luego la categoría (submuestra de 50 mensajes)
instruccion_cot = "First, explain your reasoning step by step. Then give the category. Respond in JSON."
cot = clasificar(pequeña["text"], esquema=ESQUEMA_COT, instruccion=instruccion_cot)
print(cot["categoria"].value_counts().to_dict())
print("\nUn razonamiento de ejemplo:\n", cot["razonamiento"].iloc[0][:600])

50 mensajes en 82 s (1.64 s por mensaje)
{'TOXIC': 27, 'NONTOXIC': 23}

Un razonamiento de ejemplo:
 Análisis del texto:
1. La expresión 'Orgullo del pueblo' sugiere una sensación de orgullo y pertenencia, lo cual es común en contextos de identidad política.
2. La adición de 'Republicano Argentino' especifica la afiliación política del hablante.
3. El uso de '!!!!!!' es un signo de exaltación y énfasis, lo cual puede indicar entusiasmo o apoyo.
4. El hashtag '#17AJuntosContraLaImpunidad' sugiere que el hablante está apoyando una causa o movimiento político específico.
5. La combinación de estos elementos sugiere un comentario que expresa orgullo y apoyo a una causa política.
6. Sin embargo, n


**Para recordar.** Aun con `temperature=0` y una semilla fija, las respuestas largas pueden variar entre ejecuciones idénticas: al preparar este notebook, la cadena de razonamiento cambió la categoría de 1 de los 50 mensajes entre ejecuciones repetidas en la misma GPU, mientras que las respuestas cortas (*zero-shot* y *few-shot*) se repitieron exactamente. Por eso se registran el modelo, la versión y las opciones, y se mide la estabilidad (sección 4).

## 4. Estabilidad del *prompt*

**▶ En vivo.** Un buen anotador debería dar la misma respuesta si la instrucción se redacta de otra forma. Barrie, Palaiologou y Törnberg (2024) proponen medir esa **estabilidad** con el α de Krippendorff:
- **entre *prompts*:** la misma tarea con instrucciones parafraseadas;
- **dentro de un *prompt*:** la misma instrucción repetida con temperatura mayor que 0 y otras semillas.

La estabilidad es una condición **necesaria pero no suficiente**: un modelo puede ser muy estable y estar equivocado.

In [9]:
## Paráfrasis de la instrucción (entre prompts) y repeticiones con temperatura 0,7 (dentro del prompt)
PARAFRASIS = {
    "original": SISTEMA,
    "paráfrasis 1": ("Decide whether the following social media message is TOXIC or NONTOXIC. "
                     "A TOXIC message is rude, disrespectful or unreasonable and would probably make "
                     "someone abandon the conversation. A NONTOXIC message is civil or friendly."),
    "paráfrasis 2 (español)": ("Clasifica el comentario como TOXIC o NONTOXIC. TOXIC: comentarios groseros, "
                               "irrespetuosos o poco razonables que probablemente hagan que alguien abandone "
                               "la discusión. NONTOXIC: comentarios civilizados o amables que no desalientan "
                               "la conversación."),
}
estabilidad = {}
for nombre, sistema in PARAFRASIS.items():
    estabilidad[nombre] = clasificar(pequeña["text"], sistema=sistema)["categoria"]
for semilla in [1, 2]:
    estabilidad[f"original, temperatura 0,7, semilla {semilla}"] = clasificar(
        pequeña["text"], opciones={"temperature": 0.7, "seed": semilla})["categoria"]


def alfa_krippendorff(codificaciones):
    ## α nominal para datos completos: filas = unidades, columnas = codificaciones
    valores = np.asarray(codificaciones)
    categorias = np.unique(valores)
    m = valores.shape[1]
    o = np.zeros((len(categorias), len(categorias)))
    for fila in valores:
        cuenta = np.array([(fila == c).sum() for c in categorias])
        o += (np.outer(cuenta, cuenta) - np.diag(cuenta)) / (m - 1)
    n_c = o.sum(axis=1)
    n = n_c.sum()
    desacuerdo_obs = o.sum() - np.trace(o)
    desacuerdo_esp = (n * n - (n_c ** 2).sum()) / (n - 1)
    return 1 - desacuerdo_obs / desacuerdo_esp if desacuerdo_esp > 0 else 1.0


tabla_est = pd.DataFrame(estabilidad)
entre = ["original", "paráfrasis 1", "paráfrasis 2 (español)"]
dentro = ["original", "original, temperatura 0,7, semilla 1", "original, temperatura 0,7, semilla 2"]
print(f"α de Krippendorff entre prompts:     {alfa_krippendorff(tabla_est[entre].values):.3f}")
print(f"α de Krippendorff dentro del prompt: {alfa_krippendorff(tabla_est[dentro].values):.3f}")
print("Proporción de TOXIC por versión:", (tabla_est == "TOXIC").mean().round(2).to_dict())

50 mensajes en 8 s (0.15 s por mensaje)


50 mensajes en 9 s (0.19 s por mensaje)


50 mensajes en 9 s (0.17 s por mensaje)


50 mensajes en 8 s (0.15 s por mensaje)


50 mensajes en 8 s (0.15 s por mensaje)
α de Krippendorff entre prompts:     0.235
α de Krippendorff dentro del prompt: 0.825
Proporción de TOXIC por versión: {'original': 0.68, 'paráfrasis 1': 0.98, 'paráfrasis 2 (español)': 0.88, 'original, temperatura 0,7, semilla 1': 0.6, 'original, temperatura 0,7, semilla 2': 0.68}


**Para recordar.**
- Si el α es alto, el modelo responde casi lo mismo aunque cambie la redacción o el azar. Si es bajo, la anotación depende de **cómo** se preguntó, y hay que informarlo.
- Aquí el modelo es **estable frente al azar** (la misma instrucción con temperatura 0,7), pero **inestable frente a la redacción**: las paráfrasis cambian mucho cuántos mensajes se marcan como tóxicos. Un cambio pequeño en la instrucción puede mover la prevalencia estimada, y eso es lo que hay que medir e informar.

## 5. Validación contra la codificación humana

**▶ En vivo.** Comparamos cada estrategia con `coder_1` (F1 macro, precisión y recall de la clase tóxica, κ de Cohen) y con la referencia humana (`coder_2`) en los mismos mensajes. Los mensajes `REVISAR` se informan aparte.

In [10]:
## Tabla de validación: cada estrategia frente a coder_1, con la referencia humana
def validar(nombre, categorias, datos):
    validas = categorias != "REVISAR"
    pred = categorias[validas].map(ETIQUETAS).to_numpy()
    real = datos["coder_1"].to_numpy()[validas.to_numpy()]
    return {"estrategia": nombre, "mensajes": len(categorias), "REVISAR": int((~validas).sum()),
            "F1 macro": f1_score(real, pred, average="macro"),
            "precisión (tóxico)": precision_score(real, pred), "recall (tóxico)": recall_score(real, pred),
            "κ de Cohen": cohen_kappa_score(real, pred), "% TOXIC predicho": 100 * pred.mean()}


filas = [validar("Zero-shot", zero["categoria"], muestra),
         validar("Few-shot (4 ejemplos)", few["categoria"], muestra),
         validar("Cadena de razonamiento (50)", cot["categoria"], pequeña)]
for nombre, datos in [("Referencia humana (coder_2)", muestra)]:
    humano = datos["coder_2"].map({1: "TOXIC", 0: "NONTOXIC"})
    filas.append(validar(nombre, humano.reset_index(drop=True), datos))
validacion = pd.DataFrame(filas).set_index("estrategia")
validacion.round(3)

,mensajes,REVISAR,F1 macro,precisión (tóxico),recall (tóxico),κ de Cohen,% TOXIC predicho
estrategia,,,,,,,
Zero-shot,100,0,0.711,0.703,0.833,0.427,64.0
Few-shot (4 ejemplos),100,0,0.780,0.864,0.704,0.564,44.0
Cadena de razonamiento (50),50,0,0.718,0.741,0.741,0.436,54.0
Referencia humana (coder_2),100,0,0.990,1.000,0.981,0.980,53.0


**Para recordar.**
- Un LLM pequeño, sin ejemplos, ya clasifica razonablemente. Pero tiende a **marcar demasiados mensajes como tóxicos** (recall alto, precisión menor), el mismo patrón que González-Bustamante (2024) encontró en la mayoría de los LLMs.
- Los **ejemplos reales** del entrenamiento ayudan: corrigen en parte la tendencia a sobremarcar. En 2025, con dos ejemplos inventados y extremos, el *few-shot* había empeorado el resultado.
- La **cadena de razonamiento no ayuda** a un modelo de este tamaño, y es mucho más lenta: cada respuesta es un texto largo.
- Compare con S12 y el notebook 5 (en otra muestra, así que solo como orden de magnitud): el LLM no se entrenó con ninguna etiqueta y queda cerca de TF-IDF, pero por debajo de un modelo ajustado para la tarea que se valide fuera del dominio.
- **Protocolo:** fijar el modelo, la versión y las opciones; medir contra un patrón oro humano; informar F1, precisión, recall y κ; medir la estabilidad del *prompt*; revisar los errores; y registrar los casos no legibles.

In [11]:
## Registro para la reproducibilidad: qué se usó exactamente
registro = {"modelo": MODELO, "digest": next(m.digest for m in ollama.list().models if m.model.startswith(MODELO))[:12],
            "parámetros": detalle.details.parameter_size, "cuantización": detalle.details.quantization_level,
            "servidor Ollama": version_servidor, "cliente ollama": version("ollama"), "opciones": OPCIONES,
            "mensajes": len(muestra), "semilla de la muestra": 86}
registro

{'modelo': 'llama3.2',
 'digest': 'a80c4f17acd5',
 'parámetros': '3.2B',
 'cuantización': 'Q4_K_M',
 'servidor Ollama': '0.40.7',
 'cliente ollama': '0.6.3',
 'opciones': {'temperature': 0, 'seed': 86},
 'mensajes': 100,
 'semilla de la muestra': 86}

## 6. Ejercicios

1. **Más mensajes.** Cambien `N_MENSAJES` a 200 (toda la prueba) y repitan las secciones 1 y 5. ¿Cambian las conclusiones?
2. **Otros ejemplos.** Cambien los ejemplos de la sección 2 (otra semilla, o 8 ejemplos). ¿Qué tan sensible es el *few-shot* a los ejemplos elegidos?
3. **Instrucciones en español.** Usen la paráfrasis en español como instrucción principal en la sección 1. ¿Cambian el F1 y la proporción de tóxicos?
4. **Otro modelo.** Prueben `ollama pull llama3.2:1b` (1.000 millones de parámetros) y cambien `MODELO`. ¿Cuánto se pierde con un modelo más pequeño? ¿Cuánto más rápido es?

**Referencias**
- Barrie, C., Palaiologou, E. y Törnberg, P. (2024). Prompt stability scoring for text annotation with large language models. arXiv:2407.02039.
- Gilardi, F., Alizadeh, M. y Kubli, M. (2023). ChatGPT outperforms crowd workers for text-annotation tasks. *PNAS*, 120(30), e2305016120.
- González-Bustamante, B. (2024). Benchmarking LLMs in political content text-annotation: Proof-of-concept with toxicity and incivility data. arXiv:2409.09741.
- Meta (2024). *Llama 3.2 model card*. https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct
- Wei, J. et al. (2022). Chain-of-thought prompting elicits reasoning in large language models. *Advances in Neural Information Processing Systems*, 35.